# 09 — Multimodal Fusion (Vision + Language)
### Starter Notebook

**Learning objectives**
- Understand ViT-style patch embeddings for images
- Learn how visual tokens are fused with text tokens
- Implement cross-modal attention
- Understand how Gemini processes images natively

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — Images as Token Sequences

A **Vision Transformer (ViT)** treats an image just like a sequence of tokens:
1. Split image into $P \times P$ patches
2. Linearly project each patch to a $d_{model}$-dimensional vector
3. Feed the patch tokens through transformer layers

For a 224×224 image with 16×16 patches: $\frac{224}{16} \times \frac{224}{16} = 196$ tokens.

### Exercise A1 — Implement PatchEmbedding

In [ ]:
class MyPatchEmbedding(nn.Module):
    """
    Convert an image to a sequence of patch embeddings.

    Approach: use a Conv2d with kernel_size=patch_size, stride=patch_size
              — equivalent to a linear projection of each non-overlapping patch.

    Args:
        image_size  : height/width of image (assumes square)
        patch_size  : size of each patch
        in_channels : number of image channels (3 for RGB)
        d_model     : output embedding dimension
    """

    def __init__(self, image_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        assert image_size % patch_size == 0
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_size  = patch_size

        # TODO: Conv2d projection (kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, channels, H, W)
        Returns:
            patches: (batch, num_patches, d_model)
        """
        # TODO: project, flatten spatial dims, transpose to (batch, patches, d_model)
        pass


patch_embed = MyPatchEmbedding(image_size=32, patch_size=4, d_model=64)
image = torch.randn(2, 3, 32, 32)   # tiny 32×32 image for testing
patches = patch_embed(image)
if patches is not None:
    print(f'Image shape  : {image.shape}')
    print(f'Patches shape: {patches.shape}   # expect (2, 64, 64)')
    print(f'Num patches  : {patch_embed.num_patches}   # (32/4)² = 64')
else:
    print('Not implemented yet.')

### Exercise A2 — Visualise patches

In [ ]:
# Create a synthetic coloured image and show how it splits into patches
H, W, P = 16, 16, 4   # 16×16 image, 4×4 patches → 16 patches
img = torch.zeros(3, H, W)
# Colour each 4×4 patch differently
colours = plt.cm.tab20.colors
for i in range(H // P):
    for j in range(W // P):
        patch_id = i * (W // P) + j
        r, g, b = colours[patch_id % 20]
        img[:, i*P:(i+1)*P, j*P:(j+1)*P] = torch.tensor([r, g, b]).view(3, 1, 1)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img.permute(1, 2, 0).numpy())
axes[0].set_title(f'{H}×{W} image')
axes[0].grid(True, which='both', color='white', linewidth=2)
axes[0].set_xticks(range(0, W, P)); axes[0].set_yticks(range(0, H, P))

axes[1].text(0.5, 0.5,
    f'{H}×{W} image\n→ {(H//P)*(W//P)} patches\n(each {P}×{P} pixels)\n→ {(H//P)*(W//P)} tokens',
    ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
axes[1].axis('off')

plt.suptitle('Image → Patch Tokens', fontsize=13)
plt.tight_layout(); plt.show()

## Part B — Fusion Strategy: Concatenation

The simplest way to combine image and text: **prepend** visual tokens to the text token sequence, then run a standard transformer.

```
[IMG_1] [IMG_2] ... [IMG_N] [TEXT_1] [TEXT_2] ... [TEXT_M]
      ↑ visual tokens ↑            ↑ text tokens ↑
```

The text tokens can attend to visual tokens and vice versa — a fully shared attention space.

In [ ]:
from src.models.multimodal import PatchEmbedding
from src.models.attention import MultiHeadAttention

d_model = 64

# Visual encoder
patch_emb = PatchEmbedding(image_size=32, patch_size=8, d_model=d_model)
image = torch.randn(2, 3, 32, 32)
visual_tokens = patch_emb(image)           # (2, 16, 64)  — 16 patches

# Text tokens (from embedding)
text_embed = nn.Embedding(100, d_model)
text_ids = torch.randint(0, 100, (2, 10))  # batch=2, 10 text tokens
text_tokens = text_embed(text_ids)         # (2, 10, 64)

# TODO: concatenate visual + text tokens and run through attention
# multimodal_tokens = ...

print(f'Visual tokens : {visual_tokens.shape}')
print(f'Text tokens   : {text_tokens.shape}')
print('[Exercise B] Concatenate and run through MultiHeadAttention.')

## Part C — Cross-Modal Attention

An alternative: separate visual and language towers with dedicated **cross-attention** bridges.
Text queries attend to visual keys/values.

### Exercise C1

In [ ]:
from src.models.attention import MultiHeadAttention

class CrossModalAttention(nn.Module):
    """
    Cross-modal attention: text queries attend to image keys/values.

    Q = from text, K/V = from image
    """

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        # TODO: a single MultiHeadAttention layer
        # Note: Q comes from text, K/V come from image — no causal mask

    def forward(self, text: torch.Tensor, image: torch.Tensor) -> torch.Tensor:
        """
        Args:
            text  : text tokens  (batch, text_len,  d_model)
            image : image tokens (batch, image_len, d_model)
        Returns:
            text updated with visual context  (batch, text_len, d_model)
        """
        # TODO: Q=text, K=image, V=image
        pass


cross_attn = CrossModalAttention(d_model=64, n_heads=4)
result = cross_attn(text_tokens, visual_tokens)
if result is not None:
    print(f'Cross-attention output: {result.shape}   # expect (2, 10, 64)')
else:
    print('Not implemented yet.')

## Part D — Library Vision Encoder

In [ ]:
from src.models.multimodal import VisionEncoder, MultiModalFusion

# Vision encoder: ViT-style
vision_enc = VisionEncoder(
    image_size=32, patch_size=8,
    d_model=64, n_heads=4, n_layers=2
)
image = torch.randn(2, 3, 32, 32)
visual_features = vision_enc(image)
print(f'VisionEncoder output: {visual_features.shape}')  # (2, num_patches, 64)

# Multimodal fusion
fusion = MultiModalFusion(d_model=64, n_heads=4)
text_features = torch.randn(2, 10, 64)
fused = fusion(text_features, visual_features)
print(f'Fused output: {fused.shape}')   # (2, 10, 64) — text updated with visual context

## Summary

| Strategy | How | Pros | Cons |
|----------|-----|------|------|
| Token concat | Prepend image tokens to text | Simple, full bidirectional | Long sequences, expensive |
| Cross-attention | Text queries image K/V | Separate towers, efficient | More complex architecture |
| Prefix tokens | Compressed visual representation | Short & fast | Information compression loss |

**Gemini** uses a native multimodal approach where images are encoded as discrete tokens in the same vocabulary space as text — no separate fusion step.

**Next:** `../part4_training/10_tokenization_starter.ipynb` — turning raw text into token IDs.